In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
print("Imports OK")

Imports OK


In [2]:
df = pd.read_csv("train_data_clean.csv")
print("Loaded: train_data_clean.csv")
print("Shape:", df.shape)
display(df.head())

Loaded: train_data_clean.csv
Shape: (53991, 9)


,Date of Sale (dd/mm/yyyy),Sale Year,Sale Month,is_apartment,location,Description of Property,Not Full Market Price,Price_Adjusted_VAT_Clamped,vat_adjusted_flag
0,2016-09-30,2016,9,0,Cork,Second-Hand Dwelling house /Apartment,No,181000.00000,0
1,2016-12-20,2016,12,0,Cork,New Dwelling house /Apartment,No,60000.00000,1
2,2016-09-28,2016,9,1,Wexford,New Dwelling house /Apartment,No,70565.00435,1
3,2016-09-16,2016,9,0,Wicklow,New Dwelling house /Apartment,No,253499.98000,1
4,2016-01-29,2016,1,0,Dublin 27,Second-Hand Dwelling house /Apartment,No,310000.00000,0


In [3]:
target = "Price_Adjusted_VAT_Clamped"

num_features = ["Sale Year", "Sale Month", "is_apartment", "vat_adjusted_flag"]
cat_features  = ["location", "Description of Property", "Not Full Market Price"]

X_num = df[num_features].copy()
X_cat = df[cat_features].copy()
y = df[target].copy()

# --- One-Hot Encoding ---
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat_encoded = encoder.fit_transform(X_cat)
cat_col_names = encoder.get_feature_names_out(cat_features)

X = pd.concat([
    X_num.reset_index(drop=True),
    pd.DataFrame(X_cat_encoded, columns=cat_col_names)
], axis=1)

# --- 标准化（Ridge/Lasso 对特征尺度敏感，必须标准化）---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

X shape: (53991, 60)
Train: (43192, 60), Test: (10799, 60)


In [4]:
# --- RidgeCV ---
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000]

ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)
print(f"Ridge best alpha: {ridge_cv.alpha_}")

# --- LassoCV ---
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train, y_train)
print(f"Lasso best alpha: {lasso_cv.alpha_}")

Ridge best alpha: 10.0
Lasso best alpha: 100.0


In [5]:
# --- Ridge ---
y_pred_ridge_train = ridge_cv.predict(X_train)
y_pred_ridge_test  = ridge_cv.predict(X_test)

r2_ridge_train  = r2_score(y_train, y_pred_ridge_train)
r2_ridge_test   = r2_score(y_test,  y_pred_ridge_test)
rmse_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test))
mae_ridge_test  = mean_absolute_error(y_test, y_pred_ridge_test)

# --- Lasso ---
y_pred_lasso_train = lasso_cv.predict(X_train)
y_pred_lasso_test  = lasso_cv.predict(X_test)

r2_lasso_train  = r2_score(y_train, y_pred_lasso_train)
r2_lasso_test   = r2_score(y_test,  y_pred_lasso_test)
rmse_lasso_test = np.sqrt(mean_squared_error(y_test, y_pred_lasso_test))
mae_lasso_test  = mean_absolute_error(y_test, y_pred_lasso_test)

# --- 对比 ---
print("=== Model Comparison (Internal Test Set, 20%) ===")
print(f"\n{'Model':25s} {'R² Train':>10s} {'R² Test':>10s} {'RMSE':>12s} {'MAE':>12s}")
print("-" * 72)
print(f"{'Linear Regression':25s} {'0.4732':>10s} {'0.4702':>10s} {'126,864':>12s} {'--':>12s}")
print(f"{'Ridge (alpha=10)':25s} {r2_ridge_train:>10.4f} {r2_ridge_test:>10.4f} {rmse_ridge_test:>12,.0f} {mae_ridge_test:>12,.0f}")
print(f"{'Lasso (alpha=100)':25s} {r2_lasso_train:>10.4f} {r2_lasso_test:>10.4f} {rmse_lasso_test:>12,.0f} {mae_lasso_test:>12,.0f}")

# --- Lasso 特征选择效果 ---
n_zero = int((lasso_cv.coef_ == 0).sum())
print(f"\nLasso zeroed out {n_zero} / {len(lasso_cv.coef_)} features")

=== Model Comparison (Internal Test Set, 20%) ===

Model                       R² Train    R² Test         RMSE          MAE
------------------------------------------------------------------------
Linear Regression             0.4732     0.4702      126,864           --
Ridge (alpha=10)              0.4732     0.4702      126,864       93,891
Lasso (alpha=100)             0.4731     0.4701      126,875       93,917

Lasso zeroed out 4 / 60 features


In [6]:
import re

date_col  = "Date of Sale (dd/mm/yyyy)"
price_col = "Price (€)"
VAT_RATE  = 0.135

p05 = df["Price_Adjusted_VAT_Clamped"].quantile(0.05)
p95 = df["Price_Adjusted_VAT_Clamped"].quantile(0.95)

def build_location(row):
    county   = str(row["County"]).strip() if pd.notna(row["County"]) else ""
    district = row["dublin_district"]
    if county == "Dublin":
        if pd.notna(district) and str(district).strip() != "":
            return "Dublin " + str(district).strip()
        else:
            return "Dublin Other"
    else:
        return county if county else "Unknown"

# --- Load ---
df_test_raw = pd.read_csv("test_data.csv")
print("Loaded: test_data.csv, Shape:", df_test_raw.shape)

df_test = df_test_raw.copy()

# --- Text normalization ---
for c in [col for col in df_test.columns if df_test[col].dtype == "object"]:
    df_test[c] = df_test[c].apply(lambda x: re.sub(r"\s+", " ", str(x).strip()) if pd.notna(x) else x)

# --- Date & Price parsing ---
df_test[date_col] = pd.to_datetime(df_test[date_col], dayfirst=True, errors="coerce")
df_test[price_col] = pd.to_numeric(
    df_test[price_col].astype(str).str.replace("€", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

# --- Time features ---
df_test["Sale Year"]  = df_test[date_col].dt.year
df_test["Sale Month"] = df_test[date_col].dt.month

# --- is_apartment ---
df_test["is_apartment"] = df_test["Address"].str.contains(
    r"apartment|apt|unit|flat", case=False, na=False
).astype(int)

# --- location ---
df_test["location"] = df_test.apply(build_location, axis=1)

# --- Drop duplicates + filter invalid ---
df_test = df_test.drop_duplicates()
reject_mask = (
    df_test[price_col].isna() | (df_test[price_col] <= 0) |
    df_test[date_col].isna() |
    df_test["County"].isna() | (df_test["County"].astype(str).str.strip() == "")
)
df_test = df_test.loc[~reject_mask].copy()

# --- VAT adjustment ---
cond_test = (
    (df_test["Description of Property"].astype(str).str.strip() == "New Dwelling house /Apartment") &
    (df_test["VAT Exclusive"].astype(str).str.strip().str.lower() == "yes") &
    (df_test[price_col].notna())
)
df_test["Price_Adjusted_VAT"] = df_test[price_col].copy()
df_test.loc[cond_test, "Price_Adjusted_VAT"] = df_test.loc[cond_test, price_col] * (1 + VAT_RATE)
df_test["vat_adjusted_flag"] = cond_test.astype(int)

# --- Price clamping ---
df_test["Price_Adjusted_VAT_Clamped"] = df_test["Price_Adjusted_VAT"].clip(lower=p05, upper=p95)

print(f"Test shape after cleaning: {df_test.shape}")

Loaded: test_data.csv, Shape: (10000, 14)
Test shape after cleaning: (9999, 21)


In [7]:
# --- Build test features ---
X_test_num = df_test[num_features].reset_index(drop=True)
X_test_cat = df_test[cat_features].reset_index(drop=True)

X_test_cat_encoded = encoder.transform(X_test_cat)
X_test_final = pd.concat([
    X_test_num,
    pd.DataFrame(X_test_cat_encoded, columns=cat_col_names)
], axis=1)

# 使用训练集的 scaler 标准化
X_test_scaled = scaler.transform(X_test_final)

y_test_final = df_test["Price_Adjusted_VAT_Clamped"].values

# --- Predict ---
y_pred_ridge = ridge_cv.predict(X_test_scaled)
y_pred_lasso = lasso_cv.predict(X_test_scaled)

# --- Metrics ---
def get_metrics(y_true, y_pred):
    return {
        "R²":   r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE":  mean_absolute_error(y_true, y_pred)
    }

m_ridge = get_metrics(y_test_final, y_pred_ridge)
m_lasso = get_metrics(y_test_final, y_pred_lasso)

print("=== Final Model Comparison (External Test Set) ===")
print(f"\n{'Model':25s} {'R²':>8s} {'RMSE':>12s} {'MAE':>12s}")
print("-" * 60)
print(f"{'Linear Regression':25s} {'0.3976':>8s} {'136,947':>12s} {'104,861':>12s}")
print(f"{'Ridge (alpha=10)':25s} {m_ridge['R²']:>8.4f} {m_ridge['RMSE']:>12,.0f} {m_ridge['MAE']:>12,.0f}")
print(f"{'Lasso (alpha=100)':25s} {m_lasso['R²']:>8.4f} {m_lasso['RMSE']:>12,.0f} {m_lasso['MAE']:>12,.0f}")
print(f"{'Random Forest':25s} {'0.2834':>8s} {'149,331':>12s} {'113,972':>12s}")

=== Final Model Comparison (External Test Set) ===

Model                           R²         RMSE          MAE
------------------------------------------------------------
Linear Regression           0.3976      136,947      104,861
Ridge (alpha=10)            0.3975      136,922      104,848
Lasso (alpha=100)           0.3972      136,959      104,896
Random Forest               0.2834      149,331      113,972
